## Reasoning Patch

In [1]:
%load_ext autoreload
%autoreload 2

### Overview

Intervenes on a single sequence position and single layer, with or without attention pattern freezing. Record the effect on one variable.

### Set-up

Imports `_config`/`_dataset`/`_prompt`/`_mapping` plus the multitoken intervention primitives used to patch the single fixed layer below.

In [ ]:
import torch
import gc
from tqdm import tqdm

import sys
sys.path.append("src")
import _config
import _dataset
import _prompt
import _mapping
from _intervention import prepare_batch_multitoken_intervention, batch_intervene

## Experiment Config

Builds a `PromptConfig` (`GPT-OSS_stepwise`, `h`), an `InterventionConfig` with `intervention_loc="intervened_pattern"`, a fixed `intervention_ids=[27]` and `layers=[0]`, an *enabled* `AttentionFreezeConfig` (`num_attention=20`, `divide_num=100`), and a `RunConfig` writing to `intervened_pattern/` with `filename_order="location_prompt"`; resolves `intervention_ids`/`tok_pos_list` via `_config.resolve_intervention_ids`/`build_tok_pos_list`.

In [3]:
prompt_config = _config.PromptConfig(
    model_type="GPT-OSS_stepwise", # GPT-OSS or R1
    prompt_type="h", # {null / h / h1 / h2}_{null / pre_result / pre_final_sum / ...}
)
intervention_config = _config.InterventionConfig(
    intervention_loc="intervened_pattern", # restatement or reasoning or restatement_and_reasoning
    intervention_ids=[27],
    tok_pos_fn=_mapping.intervene_id_to_tok_pos_stepwise_3_digit_h,
    layers=[0],
)
attention_config = _config.AttentionFreezeConfig(
    enabled=True,
    num_attention=20,
    dataset_fn=_dataset.create_h_dataset,
    num_digits=3,
    prompt_fn=_prompt.get_stepwise_prompt,
    divide_num=100,
)
run_config = _config.RunConfig(
    result_dir="intervened_pattern",
    filename_order="location_prompt",
)

intervention_ids = _config.resolve_intervention_ids(
    prompt_config.model_type,
    prompt_config.prompt_type,
    intervention_config.intervention_loc,
    intervention_config.intervention_ids,
)
tok_pos_list = _config.build_tok_pos_list(intervention_config.tok_pos_fn, intervention_ids)
modifier_fn = lambda prompt, add_ds_entry: prompt

## Set up Experiment

Loads the GPT-OSS_stepwise model, defines `intervene_on_attention_prompt` (splices the tested `intervention_ids` content into the auxiliary attention-freeze prompts) and builds the 20 frozen-attention hooks from those prompts via `_config.build_attention_freeze_hooks`, then loads the `h` prompts to patch.

In [4]:
model, tokenizer = _config.load_model(prompt_config.model_type)

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

In [5]:
def intervene_on_attention_prompt(prompt, add_ds_entry):
    source_prompt = attention_config.prompt_fn(
        add_ds_entry["source_1_digits"],
        add_ds_entry["source_2_digits"],
        add_ds_entry["source_1_num"],
        add_ds_entry["source_2_num"],
    )
    return _prompt.get_intervened_prompt(intervention_ids, prompt, source_prompt)

modifier_fn = intervene_on_attention_prompt

In [6]:
attention_freeze_hooks = _config.build_attention_freeze_hooks(
    model,
    tokenizer,
    attention_config,
    modifier_fn=modifier_fn,
)

<|start|>system<|message|>You are ChatGPT, a large language model trained by OpenAI.
Knowledge cutoff: 2024-06
Current date: 2025-06-28

Reasoning: high

# Valid channels: analysis, commentary, final. Channel must be included for every message.<|end|><|start|>developer<|message|># Instruction: In analysis, add two numbers stepwise. In final, output only the sum.<|end|><|start|>user<|message|>What is 104+112?<|end|><|start|>assistant<|channel|>analysis<|message|>The user asks: "What is 104+112?" The instruction from the developer says: "In the analysis, add two numbers stepwise. For the final output, output only the sum." So we need to do the addition stepwise in the analysis, and then in the final output, just output the sum. So I need to do stepwise addition in analysis, then output only the sum in final. So in analysis, I will show the stepwise addition: 104 + 112. Let's do it: 104 + 112 = 104 + 100 + 10 + 2 = 104 + 100 = 204, 204 + 10 = 214, 214 + 2 = 216. So the sum is 216. In fina

In [7]:
prompts = _config.load_prompts(prompt_config)
print(f"loaded {len(prompts)} prompts")

loaded 256 prompts


In [8]:
print(intervention_ids)

[27]


In [9]:
# for i, row in prompts.iterrows():
#     print(list(enumerate(tokenizer.convert_ids_to_tokens(tokenizer(row["base_prompt"], add_special_tokens=False, return_tensors="pt")["input_ids"][0]))))

In [10]:
print(tok_pos_list)

[265]


## Run Experiment

Intervene on a single sequence position and a single layer. If attention_config is enabled, copy the prompt 20 times for each imported attention pattern.

In [11]:
batch_size = run_config.resolved_batch_size(attention_config.enabled)

header = list(prompts.columns) + ['intervention_ids', 'intervention_id', 'generated_text', 'factual_label_probability', 'counterfactual_label_probability']
filepath = _config.build_output_filepath(prompt_config, intervention_config, run_config, header)

for i in tqdm(range(0, len(prompts), batch_size)):
    # Preparing prompts and labels
    batch_rows = prompts.iloc[i:i+batch_size]
    factual_labels, counterfactual_labels = _config.prepare_label_tensors(tokenizer, batch_rows)

    base_prompts = batch_rows['base_prompt'].tolist()
    source_prompts = batch_rows['source_prompt'].tolist()

    if attention_config.enabled:
        base_prompts = base_prompts * attention_config.num_attention
        source_prompts = source_prompts * attention_config.num_attention
        factual_labels = factual_labels.repeat(attention_config.num_attention)
        counterfactual_labels = counterfactual_labels.repeat(attention_config.num_attention)

    # Preparing intervention hooks
    intervene_hooks = []
    tokens, source_tokens, hooks = prepare_batch_multitoken_intervention(
        model,
        tokenizer,
        intervention_config.layers,
        tok_pos_list,
        base_prompts,
        source_prompts,
        module_format=intervention_config.module_format,
        pre_hook=intervention_config.pre_hook,
    )
    intervene_hooks += hooks
    input_length = tokens["input_ids"].shape[1]

    # Forward pass
    with torch.no_grad():
        output = batch_intervene(model, tokens["input_ids"], intervene_hooks+attention_freeze_hooks, attention_mask=tokens["attention_mask"])
    pred_toks = output.logits[:,-1,:].argmax(dim=-1)
    prob = torch.nn.functional.softmax(output.logits[:,-1,:], dim=-1)
    factual_prob = prob[torch.arange(prob.shape[0]), factual_labels]
    counterfactual_prob = prob[torch.arange(prob.shape[0]), counterfactual_labels]
    tokens["input_ids"] = torch.cat([tokens["input_ids"], pred_toks.unsqueeze(-1)], dim=1)
    del output
    
    # Writing results
    if attention_config.enabled:
        for j in range(attention_config.num_attention):
            generated_text = tokenizer.decode(tokens["input_ids"][j,input_length:]).replace(tokenizer.pad_token[-1], "")
            _config.write_to_csv(filepath, batch_rows.iloc[0].to_list() + [intervention_ids, ', '.join(str(id) for id in intervention_ids), generated_text.replace(tokenizer.pad_token[-1], ""), factual_prob[j].item(), counterfactual_prob[j].item()])
    else:
        for j in range(len(batch_rows)):
            generated_text = tokenizer.decode(tokens["input_ids"][j,input_length:]).replace(tokenizer.pad_token[-1], "")
            _config.write_to_csv(filepath, batch_rows.iloc[j].to_list() + [intervention_ids, ', '.join(str(id) for id in intervention_ids), generated_text.replace(tokenizer.pad_token[-1], ""), factual_prob[j].item(), counterfactual_prob[j].item()])

    del tokens, source_tokens, intervene_hooks
    torch.cuda.empty_cache()
    gc.collect()

  0%|                                                                                                                       | 0/256 [00:00<?, ?it/s]

100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████| 256/256 [39:06<00:00,  9.17s/it]
